# 06 - Main Launcher (Single Node Multi-GPU)
Use this notebook to select 1/2/3 GPUs and launch `train_04.py`.

This workflow uses `torch.nn.DataParallel` configured inside `train_04.py`.

In [1]:
import os
import sys
import subprocess
from pathlib import Path
import torch
import math

ROOT = Path.cwd()
TRAIN_SCRIPT = ROOT / 'train_04.py'
print('Workspace:', ROOT)
print('Train script exists:', TRAIN_SCRIPT.exists())

Workspace: /projectnb/dl4ds/students/sxqian/qwen3_5_vision_project
Train script exists: True


In [2]:
gpu_count = torch.cuda.device_count()
print(f'Detected CUDA GPUs: {gpu_count}')
for i in range(gpu_count):
    print(f'GPU {i}: {torch.cuda.get_device_name(i)}')

Detected CUDA GPUs: 2
GPU 0: Tesla V100-SXM2-16GB
GPU 1: Tesla V100-SXM2-16GB


## Configure GPU Split
Set GPU IDs you want to use. Example: `[0, 1]` for 2 GPUs, `[0, 1, 2]` for 3 GPUs.

In [3]:
from pathlib import Path
import subprocess
import sys
import os

ROOT = Path("/projectnb/dl4ds/students/sxqian/qwen3_5_vision_project").resolve()
TRAIN_SCRIPT = ROOT / "train_04.py"      # 单卡脚本
DDP_SCRIPT = ROOT / "train_04_ddp.py"       # 多卡脚本（以后用）

print("ROOT:", ROOT)
print("Python:", sys.executable)
print("Single-GPU script:", TRAIN_SCRIPT)
print("Multi-GPU script:", DDP_SCRIPT)

ROOT: /projectnb/dl4ds/students/sxqian/qwen3_5_vision_project
Python: /projectnb/dl4ds/students/sxqian/qwen3_5_vision_project/venv/bin/python
Single-GPU script: /projectnb/dl4ds/students/sxqian/qwen3_5_vision_project/train_04.py
Multi-GPU script: /projectnb/dl4ds/students/sxqian/qwen3_5_vision_project/train_04_ddp.py


In [4]:
# ===== Launch Config =====
USE_DDP = True          # False=单卡/普通脚本；True=多卡DDP
NUM_GPUS = 2             # DDP时用几张卡
GPU_IDS = [0]            # 单卡脚本时可传 [0]；以后也可扩展
## dataset size ==68686
DATASET_SIZE =  68686
CHUNK_SIZE = 8000
MODEL_SIZE = "4b"
NUM_EPOCHS = 2
RESUME =True

print("USE_DDP:", USE_DDP)
print("NUM_GPUS:", NUM_GPUS)
print("GPU_IDS:", GPU_IDS)
print("DATASET_SIZE:", DATASET_SIZE)
print("MODEL_SIZE:", MODEL_SIZE)
print("NUM_EPOCHS:", NUM_EPOCHS)
print("RESUME:", RESUME)

USE_DDP: True
NUM_GPUS: 2
GPU_IDS: [0]
DATASET_SIZE: 22000
MODEL_SIZE: 4b
NUM_EPOCHS: 2
RESUME: True


In [5]:
def build_command():
    if USE_DDP:
        script = DDP_SCRIPT
        if not script.exists():
            raise FileNotFoundError(f"Cannot find DDP script: {script}")

        cmd = [
            sys.executable,
            "-m",
            "torch.distributed.run",
            f"--nproc_per_node={NUM_GPUS}",
            str(script),
        ]
    else:
        script = TRAIN_SCRIPT
        if not script.exists():
            raise FileNotFoundError(f"Cannot find training script: {script}")

        cmd = [
            sys.executable,
            str(script),
        ]

        if GPU_IDS:
            cmd += ["--gpu-ids", ",".join(str(x) for x in GPU_IDS)]

    if DATASET_SIZE is not None:
        cmd += ["--dataset-size", str(DATASET_SIZE)]
    if CHUNK_SIZE is not None:
        cmd += ["--chunk-size", str(CHUNK_SIZE)]
    if MODEL_SIZE is not None:
        cmd += ["--model-size", str(MODEL_SIZE)]
    if NUM_EPOCHS is not None:
        cmd += ["--num-epochs", str(NUM_EPOCHS)]
    if not RESUME:
        cmd += ["--no-resume"]

    return cmd

In [ ]:
for _ in range(math.ceil(DATASET_SIZE / CHUNK_SIZE)):
    cmd = build_command()

    print("Launching command:")
    print(" ".join(cmd))

    proc = subprocess.Popen(cmd, cwd=str(ROOT))
    return_code = proc.wait()

    print("Training exit code:", return_code)

Launching command:
/projectnb/dl4ds/students/sxqian/qwen3_5_vision_project/venv/bin/python -m torch.distributed.run --nproc_per_node=2 /projectnb/dl4ds/students/sxqian/qwen3_5_vision_project/train_04_ddp.py --dataset-size 22000 --chunk-size 8000 --model-size 4b --num-epochs 2
[main] local_rank=0, rank=0, world_size=2, device_cuda=device(type='cuda', index=0)
Checking tokenizer access for: Qwen/Qwen3.5-4B
HF token found in env: False
[main] local_rank=1, rank=1, world_size=2, device_cuda=device(type='cuda', index=1)
Checking tokenizer access for: Qwen/Qwen3.5-4B
HF token found in env: False


Tokenizer loaded successfully.
vocab_size: 248044
Checking processor access for: Qwen/Qwen3.5-4B
HF token found in env: False
Tokenizer loaded successfully.
vocab_size: 248044
Checking processor access for: Qwen/Qwen3.5-4B
HF token found in env: False
Processor loaded successfully.
Qwen init device: cuda:1
LOCAL_RANK env: 1
No previous state found. Starting from base model.
Processor loaded successfully.
Qwen init device: cuda:0
LOCAL_RANK env: 0
No previous state found. Starting from base model.


The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d
The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d
Loading weights: 100%|██████████| 723/723 [00:04<00:00, 155.14it/s]


Loaded (VL): Qwen/Qwen3.5-4B
trainable params: 10,616,832 || all params: 4,549,882,368 || trainable%: 0.2333
[setup] rank=0, set Qwen.device=cuda:0
Loaded (VL): Qwen/Qwen3.5-4B
trainable params: 10,616,832 || all params: 4,549,882,368 || trainable%: 0.2333
[setup] rank=1, set Qwen.device=cuda:1
DDP enabled on rank 0, local_rank 0, world_size 2
Model and optimizer are ready. Job index: 0
Model and optimizer are ready. Job index: 0
[dataset chunk] start=0, end=8000, chunk_size=8000
[chunk info] total_samples=22000, chunk_size=8000, total_chunks=3
[dataset chunk] start=0, end=8000, chunk_size=8000
[chunk info] total_samples=22000, chunk_size=8000, total_chunks=3
Transforming dataset of size: 7200
Using max_text_len: 384
Using prompt_max_len: 384
Using image_size: 40x250
Using image token: <|image_pad|>
Transforming dataset of size: 7200
Using max_text_len: 384
Using prompt_max_len: 384
Using image_size: 40x250
Using image token: <|image_pad|>
Transform complete.
input_ids shape: torch.Siz

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


step 100 | loss 0.0787 | time 1.687s
step 100 | loss 0.0574 | time 1.682s
step 200 | loss 0.0940 | time 1.679s
step 200 | loss 0.0041 | time 1.677s
step 300 | loss 0.0035 | time 1.851s
step 300 | loss 0.0721 | time 1.852s
step 400 | loss 0.0668 | time 1.673s
step 400 | loss 0.0082 | time 1.677s
step 500 | loss 0.0261 | time 1.692s
step 500 | loss 0.0317 | time 1.695s
step 800 | loss 0.0236 | time 1.683s
step 800 | loss 0.0462 | time 1.691s
step 900 | loss 0.0441 | time 1.876s
step 900 | loss 0.0036 | time 1.875s
step 1000 | loss 0.0380 | time 1.690s
step 1000 | loss 0.0311 | time 1.691s
step 1100 | loss 0.0097 | time 1.691s
step 1100 | loss 0.0205 | time 1.687s
step 1200 | loss 0.0324 | time 1.689s
step 1200 | loss 0.0062 | time 1.683s
step 1300 | loss 0.0122 | time 1.674s
step 1300 | loss 0.0049 | time 1.663s
step 1400 | loss 0.0655 | time 1.695s
step 1400 | loss 0.0132 | time 1.688s
step 1500 | loss 0.1007 | time 1.853s
step 1500 | loss 0.0630 | time 1.890s
step 1600 | loss 0.1271 | 

In [7]:
import os, torch
print("CUDA_VISIBLE_DEVICES =", os.environ.get("CUDA_VISIBLE_DEVICES"))
print("torch.cuda.is_available() =", torch.cuda.is_available())
print("torch.cuda.device_count() =", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i))
import os
print(os.environ.get("HF_HOME"))
print(os.environ.get("HF_DATASETS_CACHE"))

CUDA_VISIBLE_DEVICES = 0,1
torch.cuda.is_available() = True
torch.cuda.device_count() = 2
0 Tesla V100-SXM2-16GB
1 Tesla V100-SXM2-16GB
/projectnb/dl4ds/students/sxqian/cache/huggingface
/projectnb/dl4ds/students/sxqian/cache/huggingface


## SCC Notes
- Request GPUs in your SCC job (interactive or batch).
- Keep `GPU_IDS` consistent with what SCC allocates.
- Typical choices:
  - 1 GPU: `[0]`
  - 2 GPU: `[0, 1]`
  - 3 GPU: `[0, 1, 2]`